# 第二章 — 数组与序列（An Array of Sequences）

**本章包含代码片段的小节：**

* 列表推导式与生成器表达式（List Comprehensions and Generator Expressions）
* 元组不只是不可变列表（Tuples Are Not Just Immutable Lists）
* 解包序列与可迭代对象（Unpacking sequences and iterables）
* 序列的模式匹配（Pattern Matching with Sequences）
* 切片（Slicing）
* 对序列使用 + 和 *（Using + and * with Sequences）
* 序列的增量赋值（Augmented Assignment with Sequences）
* list.sort 与 sorted 内置函数（list.sort and the sorted Built-In Function）
* 列表并非总是答案（When a List Is Not the Answer）
* 内存视图（Memory Views）
* NumPy 与 SciPy
* 双端队列与其他队列（Deques and Other Queues）
* 肥皂箱（Soapbox）

## 列表推导式与生成器表达式（List Comprehensions and Generator Expressions）

#### 示例 2-1. 从字符串构建 Unicode 码位列表

In [1]:
symbols = '$¢£¥€¤'
codes = []

for symbol in symbols:
    codes.append(ord(symbol))

codes

[36, 162, 163, 165, 8364, 164]

#### 示例 2-2. 用列表推导式从字符串构建 Unicode 码位列表

In [2]:
symbols = '$¢£¥€¤'

codes = [ord(symbol) for symbol in symbols]

codes

[36, 162, 163, 165, 8364, 164]

#### 专栏：列表推导式不再泄漏其变量

In [3]:
x = 'ABC'
codes = [ord(x) for x in x]
x

'ABC'

In [4]:
codes

[65, 66, 67]

In [5]:
codes = [last := ord(c) for c in x]
last

67

#### 示例 2-3. 用列表推导式和 map/filter 组合构建同一个列表

In [6]:
symbols = '$¢£¥€¤'
beyond_ascii = [ord(s) for s in symbols if ord(s) > 127]
beyond_ascii

[162, 163, 165, 8364, 164]

In [7]:
beyond_ascii = list(filter(lambda c: c > 127, map(ord, symbols)))
beyond_ascii

[162, 163, 165, 8364, 164]

#### 示例 2-4. 用列表推导式求笛卡尔积

In [8]:
colors = ['black', 'white']
sizes = ['S', 'M', 'L']
tshirts = [(color, size) for color in colors for size in sizes]
tshirts

[('black', 'S'),
 ('black', 'M'),
 ('black', 'L'),
 ('white', 'S'),
 ('white', 'M'),
 ('white', 'L')]

In [9]:
for color in colors:
    for size in sizes:
        print((color, size))

('black', 'S')
('black', 'M')
('black', 'L')
('white', 'S')
('white', 'M')
('white', 'L')


In [10]:
shirts = [(color, size) for size in sizes
          for color in colors]
tshirts

[('black', 'S'),
 ('black', 'M'),
 ('black', 'L'),
 ('white', 'S'),
 ('white', 'M'),
 ('white', 'L')]

#### 示例 2-5. 用生成器表达式初始化元组和数组

In [11]:
symbols = '$¢£¥€¤'
tuple(ord(symbol) for symbol in symbols)

(36, 162, 163, 165, 8364, 164)

In [12]:
import array

array.array('I', (ord(symbol) for symbol in symbols))

array('I', [36, 162, 163, 165, 8364, 164])

#### 示例 2-6. 生成器表达式中的笛卡尔积

In [13]:
colors = ['black', 'white']
sizes = ['S', 'M', 'L']

for tshirt in ('%s %s' % (c, s) for c in colors for s in sizes):
    print(tshirt)

black S
black M
black L
white S
white M
white L


## 元组不只是不可变列表（Tuples Are Not Just Immutable Lists）

#### 示例 2-7. 把元组用作记录

In [14]:
lax_coordinates = (33.9425, -118.408056)
city, year, pop, chg, area = ('Tokyo', 2003, 32_450, 0.66, 8014)
traveler_ids = [('USA', '31195855'), ('BRA', 'CE342567'), ('ESP', 'XDA205856')]

for passport in sorted(traveler_ids):
    print('%s/%s' % passport)

BRA/CE342567
ESP/XDA205856
USA/31195855


In [15]:
for country, _ in traveler_ids:
    print(country)

USA
BRA
ESP


### 元组作为不可变列表（Tuples as Immutable Lists）

In [16]:
a = (10, 'alpha', [1, 2])
b = (10, 'alpha', [1, 2])
a == b

True

In [17]:
b[-1].append(99)
a == b

False

In [18]:
b

(10, 'alpha', [1, 2, 99])

In [19]:
def fixed(o):
    try:
        hash(o)
    except TypeError:
        return False
    return True


tf = (10, 'alpha', (1, 2))  # Contains no mutable items
tm = (10, 'alpha', [1, 2])  # Contains a mutable item (list)
fixed(tf)

True

In [20]:
fixed(tm)

False

## 解包序列与可迭代对象（Unpacking sequences and iterables）

In [21]:
lax_coordinates = (33.9425, -118.408056)
latitude, longitude = lax_coordinates  # unpacking
latitude

33.9425

In [22]:
longitude

-118.408056

In [23]:
divmod(20, 8)

(2, 4)

In [24]:
t = (20, 8)
divmod(*t)

(2, 4)

In [25]:
quotient, remainder = divmod(*t)
quotient, remainder

(2, 4)

In [26]:
import os

_, filename = os.path.split('/home/luciano/.ssh/id_rsa.pub')
filename

'id_rsa.pub'

### 用 * 抓取多余元素（Using * to grab excess items）

In [27]:
a, b, *rest = range(5)
a, b, rest

(0, 1, [2, 3, 4])

In [28]:
a, b, *rest = range(3)
a, b, rest

(0, 1, [2])

In [29]:
a, b, *rest = range(2)
a, b, rest

(0, 1, [])

In [30]:
a, *body, c, d = range(5)
a, body, c, d

(0, [1, 2], 3, 4)

In [31]:
*head, b, c, d = range(5)
head, b, c, d

([0, 1], 2, 3, 4)

### 在函数调用和序列字面量中用 * 解包（Unpacking with * in function calls and sequence literals）

In [32]:
def fun(a, b, c, d, *rest):
    return a, b, c, d, rest


fun(*[1, 2], 3, *range(4, 7))

(1, 2, 3, 4, (5, 6))

In [33]:
*range(4), 4

(0, 1, 2, 3, 4)

In [34]:
[*range(4), 4]

[0, 1, 2, 3, 4]

In [35]:
{*range(4), 4, *(5, 6, 7)}

{0, 1, 2, 3, 4, 5, 6, 7}

### 嵌套解包（Nested unpacking）
#### 示例 2-8. 解包嵌套元组以访问经度

[02-array-seq/metro_lat_lon.py](02-array-seq/metro_lat_lon.py)

## 序列的模式匹配（Pattern Matching with Sequences）
#### 示例 2-9. 某个虚构 Robot 类的方法

In [36]:
# def handle_command(self, message):
#     match message:
#         case ['BEEPER', frequency, times]:
#             self.beep(times, frequency)
#         case ['NECK', angle]:
#             self.rotate_neck(angle)
#         case ['LED', ident, intensity]:
#             self.leds[ident].set_brightness(ident, intensity)
#         case ['LED', ident, red, green, blue]:
#             self.leds[ident].set_color(ident, red, green, blue)
#         case _:
#             raise InvalidCommand(message)

#### 示例 2-10. 解构嵌套元组——需要 Python ≥ 3.10。
[02-array-seq/match_lat_lon.py](02-array-seq/match_lat_lon.py)

In [37]:
metro_areas = [
    ('Tokyo', 'JP', 36.933, (35.689722, 139.691667)),
    ('Delhi NCR', 'IN', 21.935, (28.613889, 77.208889)),
    ('Mexico City', 'MX', 20.142, (19.433333, -99.133333)),
    ('New York-Newark', 'US', 20.104, (40.808611, -74.020386)),
    ('São Paulo', 'BR', 19.649, (-23.547778, -46.635833)),
]

def main():
    print(f'{"":15} | {"latitude":>9} | {"longitude":>9}')
    for record in metro_areas:
        match record:
            case [name, _, _, (lat, lon)] if lon <= 0:
                print(f'{name:15} | {lat:9.4f} | {lon:9.4f}')
main()

                |  latitude | longitude
Mexico City     |   19.4333 |  -99.1333
New York-Newark |   40.8086 |  -74.0204
São Paulo       |  -23.5478 |  -46.6358


### 解释器中的序列模式匹配（Pattern Matching Sequences in an Interpreter）
#### 示例 2-11. 不用 match/case 的模式匹配。
[02-array-seq/lispy/py3.9/lis.py](02-array-seq/lispy/py3.9/lis.py)

#### 示例 2-12. 用 match/case 进行模式匹配——需要 Python ≥ 3.10。
[02-array-seq/lispy/py3.10/lis.py](02-array-seq/lispy/py3.10/lis.py)

## 切片（Slicing）

### 为什么切片和 range 不包含最后一项（Why Slices and Range Exclude the Last Item）

In [38]:
l = [10, 20, 30, 40, 50, 60]

l[:2]  # split at 2

[10, 20]

In [39]:
l[2:]

[30, 40, 50, 60]

In [40]:
l[:3]  # split at 3

[10, 20, 30]

In [41]:
l[3:]

[40, 50, 60]

### 切片对象（Slice Objects）

In [42]:
s = 'bicycle'
s[::3]

'bye'

In [43]:
s[::-1]

'elcycib'

In [44]:
s[::-2]

'eccb'

#### 示例 2-13. 从平面文件发票中取行项目

In [45]:
invoice = """
0.....6.................................40........52...55........
1909 Pimoroni PiBrella                      $17.50    3    $52.50
1489 6mm Tactile Switch x20                  $4.95    2    $9.90
1510 Panavise Jr. - PV-201                  $28.00    1    $28.00
1601 PiTFT Mini Kit 320x240                 $34.95    1    $34.95
"""

SKU = slice(0, 6)
DESCRIPTION = slice(6, 40)
UNIT_PRICE = slice(40, 52)
QUANTITY = slice(52, 55)
ITEM_TOTAL = slice(55, None)

line_items = invoice.split('\n')[2:]

for item in line_items:
    print(item[UNIT_PRICE], item[DESCRIPTION])

    $17.50   imoroni PiBrella                  
     $4.95   mm Tactile Switch x20             
    $28.00   anavise Jr. - PV-201              
    $34.95   iTFT Mini Kit 320x240             
 


### 给切片赋值（Assigning to Slices）

In [46]:
l = list(range(10))
l

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]

In [47]:
l[2:5] = [20, 30]
l

[0, 1, 20, 30, 5, 6, 7, 8, 9]

In [48]:
del l[5:7]
l

[0, 1, 20, 30, 5, 8, 9]

In [49]:
l[3::2] = [11, 22]
l

[0, 1, 20, 11, 5, 22, 9]

按设计，这个示例会抛出异常::

In [50]:
try:
    l[2:5] = 100
except TypeError as e:
    print(repr(e))

TypeError('can only assign an iterable')


In [51]:
l[2:5] = [100]
l

[0, 1, 100, 22, 9]

## 对序列使用 + 和 *（Using + and * with Sequences）

In [52]:
l = [1, 2, 3]
l * 5

[1, 2, 3, 1, 2, 3, 1, 2, 3, 1, 2, 3, 1, 2, 3]

In [53]:
5 * 'abcd'

'abcdabcdabcdabcdabcd'

### 构建列表的列表（Building Lists of Lists）

#### 示例 2-14. 一个包含三个长度为 3 的列表的列表可以表示井字棋棋盘

In [54]:
board = [['_'] * 3 for i in range(3)]
board

[['_', '_', '_'], ['_', '_', '_'], ['_', '_', '_']]

In [55]:
board[1][2] = 'X'
board

[['_', '_', '_'], ['_', '_', 'X'], ['_', '_', '_']]

#### 示例 2-15. 一个包含三个对同一列表引用的列表毫无用处

In [56]:
weird_board = [['_'] * 3] * 3
weird_board

[['_', '_', '_'], ['_', '_', '_'], ['_', '_', '_']]

In [57]:
weird_board[1][2] = 'O'
weird_board

[['_', '_', 'O'], ['_', '_', 'O'], ['_', '_', 'O']]

#### 解释

In [58]:
board = []
for i in range(3):
    row = ['_'] * 3
    board.append(row)
board

[['_', '_', '_'], ['_', '_', '_'], ['_', '_', '_']]

In [59]:
board[2][0] = 'X'
board

[['_', '_', '_'], ['_', '_', '_'], ['X', '_', '_']]

## 序列的增量赋值（Augmented Assignment with Sequences）

In [60]:
l = [1, 2, 3]
idl = id(l)

In [61]:
# NBVAL_IGNORE_OUTPUT
idl

140694277263808

In [62]:
l *= 2
l

[1, 2, 3, 1, 2, 3]

In [63]:
id(l) == idl  # same list

True

In [64]:
t = (1, 2, 3)
idt = id(t)

In [65]:
# NBVAL_IGNORE_OUTPUT
idt

140694329335488

In [66]:
t *= 2
id(t) == idt  # new tuple

False

### += 赋值谜题（A += Assignment Puzzler）
#### 示例 2-16. 一个谜题

In [67]:
t = (1, 2, [30, 40])
try:
    t[2] += [50, 60]
except TypeError as e:
    print(repr(e))

TypeError("'tuple' object does not support item assignment")


#### 示例 2-17. 意外的结果：项 t2 被改变并抛出异常

In [68]:
t

(1, 2, [30, 40, 50, 60])

#### 示例 2-18. 表达式 s[a] += b 的字节码

In [69]:
import dis

dis.dis('s[a] += b')

  1           0 LOAD_NAME                0 (s)
              2 LOAD_NAME                1 (a)
              4 DUP_TOP_TWO
              6 BINARY_SUBSCR
              8 LOAD_NAME                2 (b)
             10 INPLACE_ADD
             12 ROT_THREE
             14 STORE_SUBSCR
             16 LOAD_CONST               0 (None)
             18 RETURN_VALUE


## list.sort 与 sorted 内置函数（list.sort and the sorted Built-In Function）

In [70]:
fruits = ['grape', 'raspberry', 'apple', 'banana']
sorted(fruits)

['apple', 'banana', 'grape', 'raspberry']

In [71]:
fruits

['grape', 'raspberry', 'apple', 'banana']

In [72]:
sorted(fruits, reverse=True)

['raspberry', 'grape', 'banana', 'apple']

In [73]:
sorted(fruits, key=len)

['grape', 'apple', 'banana', 'raspberry']

In [74]:
sorted(fruits, key=len, reverse=True)

['raspberry', 'banana', 'grape', 'apple']

In [75]:
fruits

['grape', 'raspberry', 'apple', 'banana']

In [76]:
fruits.sort()
fruits

['apple', 'banana', 'grape', 'raspberry']

## 列表并非总是答案（When a List Is Not the Answer）

### 数组（Arrays）

#### 示例 2-19. 创建、保存和加载一个大型浮点数数组

In [77]:
from array import array
from random import random, seed
seed(10)  # Use seed to make the output consistent

floats = array('d', (random() for i in range(10 ** 7)))
floats[-1]

0.8190492979077034

In [78]:
with open('floats.bin', 'wb') as fp:
    floats.tofile(fp)

In [79]:
floats2 = array('d')

with open('floats.bin', 'rb') as fp:
    floats2.fromfile(fp, 10 ** 7)

floats2[-1]

0.8190492979077034

In [80]:
floats2 == floats

True

### 内存视图（Memory Views）

#### 示例 2-20. 把 6 字节内存当作 1×6、2×3 和 3×2 视图处理

In [81]:
octets = array('B', range(6))
m1 = memoryview(octets)
m1.tolist()

[0, 1, 2, 3, 4, 5]

In [82]:
m2 = m1.cast('B', [2, 3])
m2.tolist()

[[0, 1, 2], [3, 4, 5]]

In [83]:
m3 = m1.cast('B', [3, 2])
m3.tolist()

[[0, 1], [2, 3], [4, 5]]

In [84]:
m2[1,1] = 22
m3[1,1] = 33
octets

array('B', [0, 1, 2, 33, 22, 5])

#### 示例 2-21. 通过修改某个字节来改变 16 位整数数组项的值

In [85]:
numbers = array('h', [-2, -1, 0, 1, 2])
memv = memoryview(numbers)
len(memv)

5

In [86]:
memv[0]

-2

In [87]:
memv_oct = memv.cast('B')
memv_oct.tolist()

[254, 255, 255, 255, 0, 0, 1, 0, 2, 0]

In [88]:
memv_oct[5] = 4
numbers

array('h', [-2, -1, 1024, 1, 2])

### NumPy

#### 示例 2-22. numpy.ndarray 中行与列的基本操作

In [89]:
import numpy as np

a = np.arange(12)
a

array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11])

In [90]:
type(a)

numpy.ndarray

In [91]:
a.shape

(12,)

In [92]:
a.shape = 3, 4
a

array([[ 0,  1,  2,  3],
       [ 4,  5,  6,  7],
       [ 8,  9, 10, 11]])

In [93]:
a[2]

array([ 8,  9, 10, 11])

In [94]:
a[2, 1]

9

In [95]:
a[:, 1]

array([1, 5, 9])

In [96]:
a.transpose()

array([[ 0,  4,  8],
       [ 1,  5,  9],
       [ 2,  6, 10],
       [ 3,  7, 11]])

#### 示例 2-22. 加载、保存和向量化操作

In [97]:
with open('floats-1M-lines.txt', 'wt') as fp:
    for _ in range(1_000_000):
        fp.write(f'{random()}\n')

In [98]:
floats = np.loadtxt('floats-1M-lines.txt')

In [99]:
floats[-3:]

array([0.06078257, 0.61741189, 0.84349987])

In [100]:
floats *= .5
floats[-3:]

array([0.03039128, 0.30870594, 0.42174994])

In [101]:
from time import perf_counter as pc

t0 = pc()
floats /= 3
(pc() - t0) < 0.01

True

In [102]:
np.save('floats-1M', floats)
floats2 = np.load('floats-1M.npy', 'r+')
floats2 *= 6

In [103]:
floats2[-3:]

memmap([0.06078257, 0.61741189, 0.84349987])

### 双端队列与其他队列（Deques and Other Queues）

#### 示例 2-23. 使用双端队列

In [104]:
import collections

dq = collections.deque(range(10), maxlen=10)
dq

deque([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])

In [105]:
dq.rotate(3)
dq

deque([7, 8, 9, 0, 1, 2, 3, 4, 5, 6])

In [106]:
dq.rotate(-4)
dq

deque([1, 2, 3, 4, 5, 6, 7, 8, 9, 0])

In [107]:
dq.appendleft(-1)
dq

deque([-1, 1, 2, 3, 4, 5, 6, 7, 8, 9])

In [108]:
dq.extend([11, 22, 33])
dq

deque([3, 4, 5, 6, 7, 8, 9, 11, 22, 33])

In [109]:
dq.extendleft([10, 20, 30, 40])
dq

deque([40, 30, 20, 10, 3, 4, 5, 6, 7, 8])

## 肥皂箱（Soapbox）

### 混合类型列表（Mixed bag lists）

In [110]:
l = [28, 14, '28', 5, '9', '1', 0, 6, '23', 19]

In [111]:
try:
    sorted(l)
except TypeError as e:
    print(repr(e))

TypeError("'<' not supported between instances of 'str' and 'int'")


### key 参数太精彩了（Key is Brilliant）

In [112]:
l = [28, 14, '28', 5, '9', '1', 0, 6, '23', 19]

sorted(l, key=int)

[0, '1', 5, 6, '9', 14, 19, '23', 28, '28']

In [113]:
sorted(l, key=str)

[0, '1', 14, 19, '23', 28, '28', 5, 6, '9']